In [ ]:
import os
from glob import glob
import pandas as pd
import nibabel as nib
from nilearn import plotting
from scipy.ndimage import center_of_mass
import numpy as np
import sys
import math

repository_path = "/Users/user/Downloads/sift2_unbiased/"
functions_path  = os.path.join(repository_path, "code", "paper_figures", "functions")
dependencies_dir  = os.path.join(repository_path, "code", "paper_figures", "dependencies")
data_dir = os.path.join(repository_path, "data", "in_vivo")

sys.path.append(os.path.abspath(functions_path))
from prepare_connectomes import prepare_connectomes

### DEFINE DATASET

In [ ]:
dataset = "Templobe_Surgery" #"HCP_Scan_Rescan" #Developing_Children #Templobe_Surgery
ntcks = "10M"
significant_only = True # turn off to display without statistical masking
results_dir = os.path.join(data_dir, dataset)

### LOAD DATA

In [ ]:
# load labels
labels = pd.read_csv(os.path.join(dependencies_dir,"fs_default.txt"), comment='#', delim_whitespace=True, header=None,names=['ID', 'Code', 'Description', 'R', 'G', 'B', 'A'])["Description"][1:86]

# Define the parcellation image in MNI
parcellation_img = nib.load(f"{dependencies_dir}/Desikan-Killiany_left_right_in_MNI.nii.gz")

# Select the full connectomes without the surg_defect label (index 86)
subset_idx = (1,85)

Define Paths

In [ ]:
# Longitudinal Difference  
fbc_differences_none_path = f"{results_dir}/effects/{ntcks}/sift2_none/fbc_differences"
fbc_differences_cross_path = f"{results_dir}/effects/{ntcks}/sift2_cross/fbc_differences"
fbc_differences_sym_path = f"{results_dir}/effects/{ntcks}/sift2_symmetric/fbc_differences"
fbc_differences_diff_path = f"{results_dir}/effects/{ntcks}/sift2_differential/fbc_differences"

# Longitudinal Stats
fbc_stats_decreases_none_path = f"{results_dir}/stats/{ntcks}/decreases/tfnbs/sift2_none/fwe_1mpvalue.csv"
fbc_stats_increases_none_path = f"{results_dir}/stats/{ntcks}/increases/tfnbs/sift2_none/fwe_1mpvalue.csv"

fbc_stats_decreases_cross_path = f"{results_dir}/stats/{ntcks}/decreases/tfnbs/sift2_cross/fwe_1mpvalue.csv"
fbc_stats_increases_cross_path = f"{results_dir}/stats/{ntcks}/increases/tfnbs/sift2_cross/fwe_1mpvalue.csv"

fbc_stats_decreases_sym_path = f"{results_dir}/stats/{ntcks}/decreases/tfnbs/sift2_symmetric/fwe_1mpvalue.csv"
fbc_stats_increases_sym_path = f"{results_dir}/stats/{ntcks}/increases/tfnbs/sift2_symmetric/fwe_1mpvalue.csv"

fbc_stats_decreases_diff_path = f"{results_dir}/stats/{ntcks}/decreases/tfnbs/sift2_differential/fwe_1mpvalue.csv"
fbc_stats_increases_diff_path = f"{results_dir}/stats/{ntcks}/increases/tfnbs/sift2_differential/fwe_1mpvalue.csv"


Pipeline 1

In [ ]:
# load connectome files
fbc_differences_none_files = glob(os.path.join(fbc_differences_none_path, "sub*.csv"))

# load connectome dicts
fbc_differences_none_dict = prepare_connectomes(fbc_differences_none_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_none = [data for data in fbc_differences_none_dict.values()]

# calculate the mean matrices
fbc_differences_mean_none = pd.DataFrame(np.mean(np.array(fbc_differences_none), axis=0))

# calculate the stdv matrices
fbc_differences_std_none = pd.DataFrame(np.std(np.array(fbc_differences_none), axis=0))

if significant_only:
    
    # Load stats
    increases = 1 - np.loadtxt(fbc_stats_increases_none_path, delimiter=",")
    decreases = 1 - np.loadtxt(fbc_stats_decreases_none_path, delimiter=",")

    # Build combined significance mask 
    mask = (increases < 0.05) | (decreases < 0.05)

    n_sig_increases = math.ceil(np.sum(increases < 0.05) / 2)
    n_sig_decreases = math.ceil(np.sum(decreases < 0.05) / 2)

    # Apply mask to all summary matrices 
    fbc_differences_mean_none = fbc_differences_mean_none.where(mask)
    fbc_differences_std_none = fbc_differences_std_none.where(mask)

    print(f"Number of significant increases: {n_sig_increases}")
    print(f"Number of significant decreases: {n_sig_decreases}")

Pipeline 2

In [ ]:
## LOAD cross CONNECTOME MATRICES
# define path
# load connectome files
fbc_differences_cross_files = glob(os.path.join(fbc_differences_cross_path, "sub*.csv"))

# load connectome dicts
fbc_differences_cross_dict = prepare_connectomes(fbc_differences_cross_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_cross = [data for data in fbc_differences_cross_dict.values()]

# calculate the mean matrices
fbc_differences_mean_cross = pd.DataFrame(np.mean(np.array(fbc_differences_cross), axis=0))

# calculate the stdv matrices
fbc_differences_std_cross = pd.DataFrame(np.std(np.array(fbc_differences_cross), axis=0))

if significant_only:
    
    # Load stats
    increases = 1 - np.loadtxt(fbc_stats_increases_cross_path, delimiter=",")
    decreases = 1 - np.loadtxt(fbc_stats_decreases_cross_path, delimiter=",")

    # Build combined significance mask 
    mask = (increases < 0.05) | (decreases < 0.05)

    n_sig_increases = math.ceil(np.sum(increases < 0.05) / 2)
    n_sig_decreases = math.ceil(np.sum(decreases < 0.05) / 2)

    # Apply mask to all summary matrices 
    fbc_differences_mean_cross = fbc_differences_mean_cross.where(mask)
    fbc_differences_std_cross = fbc_differences_std_cross.where(mask)

    print(f"Number of significant increases: {n_sig_increases}")
    print(f"Number of significant decreases: {n_sig_decreases}")

Pipeline 3

In [ ]:
## LOAD UNB CONNECTOME MATRICES
# define path

# load connectome files
fbc_differences_sym_files = glob(os.path.join(fbc_differences_sym_path, "sub*.csv"))

# load connectome dicts
fbc_differences_sym_dict = prepare_connectomes(fbc_differences_sym_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_sym = [data for data in fbc_differences_sym_dict.values()]

# calculate the mean matrices
fbc_differences_mean_sym = pd.DataFrame(np.mean(np.array(fbc_differences_sym), axis=0))

# calculate the stdv matrices
fbc_differences_std_sym = pd.DataFrame(np.std(np.array(fbc_differences_sym), axis=0))


if significant_only:
    
    # Load stats
    increases = 1 - np.loadtxt(fbc_stats_increases_sym_path, delimiter=",")
    decreases = 1 - np.loadtxt(fbc_stats_decreases_sym_path, delimiter=",")

    # Build combined significance mask 
    mask = (increases < 0.05) | (decreases< 0.05)

    n_sig_increases = math.ceil(np.sum(increases < 0.05) / 2)
    n_sig_decreases = math.ceil(np.sum(decreases < 0.05) / 2)

    # Apply mask to all summary matrices 
    fbc_differences_mean_sym = fbc_differences_mean_sym.where(mask)
    fbc_differences_std_sym = fbc_differences_std_sym.where(mask)

    print(f"Number of significant increases: {n_sig_increases}")
    print(f"Number of significant decreases: {n_sig_decreases}")

In [ ]:
## LOAD diff CONNECTOME MATRICES
# define path

# load connectome files
fbc_differences_diff_files = glob(os.path.join(fbc_differences_diff_path, "sub*.csv"))

# load connectome dicts
fbc_differences_diff_dict = prepare_connectomes(fbc_differences_diff_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_diff = [data for data in fbc_differences_diff_dict.values()]

# calculate the mean matrices
fbc_differences_mean_diff = pd.DataFrame(np.mean(np.array(fbc_differences_diff), axis=0))

# calculate the stdv matrices
fbc_differences_std_diff = pd.DataFrame(np.std(np.array(fbc_differences_diff), axis=0))


if significant_only:
    
    # Load stats (1mpvalue)
    increases = 1 - np.loadtxt(fbc_stats_increases_diff_path, delimiter=",")
    decreases = 1 - np.loadtxt(fbc_stats_decreases_diff_path, delimiter=",")

    # Build combined significance mask 
    mask = (increases < 0.05) | (decreases < 0.05)

    n_sig_increases = math.ceil(np.sum(increases < 0.05) / 2)
    n_sig_decreases = math.ceil(np.sum(decreases < 0.05) / 2)

    # Apply mask to all summary matrices 
    fbc_differences_mean_diff = fbc_differences_mean_diff.where(mask)
    fbc_differences_std_diff = fbc_differences_std_diff.where(mask)

    print(f"Number of significant increases: {n_sig_increases}")
    print(f"Number of significant decreases: {n_sig_decreases}")

PREPARE VISUALISATION

In [ ]:
# Load the parcellation image
parcellation_data = parcellation_img.get_fdata()

# Extract unique labels (excluding the background label 0)
labels = np.unique(parcellation_data)
labels = labels[labels != 0]

# Compute the center of mass for each label
label_coords = {}
for label in labels:
    coords = center_of_mass(parcellation_data == label)
    # Convert voxel coordinates to MNI coordinates
    mni_coords = nib.affines.apply_affine(parcellation_img.affine, coords)
    label_coords[label] = mni_coords
    
# Extract the coordinates and ensure they are in the correct order
coords = [label_coords[label] for label in sorted(label_coords.keys())][:86]

### PLOT EDGE-WISE FBC (+/- STATISTICAL MASKING)

In [ ]:
# Define visualisation params
edge_threshold = None
edge_vmin = -0.5
edge_vmax = 0.5
edge_cmap = "jet"
node_color = "lightgrey"

counter=1


for matrix, title in [(fbc_differences_mean_none,"Pipeline 1"), (fbc_differences_mean_cross,"Pipeline 2"),(fbc_differences_mean_sym,"Pipeline 3"),(fbc_differences_mean_diff,"Pipeline 4")]:

    # Plot the connectome
    plotting.plot_connectome(matrix, coords, edge_cmap=edge_cmap, edge_threshold = edge_threshold, edge_vmin = edge_vmin, edge_vmax = edge_vmax, node_color=node_color, title=title, radiological = True, colorbar=False, display_mode='ortho')

    counter += 1